# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 12 — Architectures, intervals, and a second dataset

Five gaps remain in the manuscript, all of them requiring computation.

The mechanism is claimed for message passing but every trained model in the attribution
analysis is a GIN, so the association with atom degree is established for one architecture
only. The comparison between agreement and predictive performance rests on twenty models,
which is too few to resolve anything. The panel of structural rankings is reported as
thirty-six point estimates with no intervals, and a claim about which ranking stays above
chance is read off them. The headline ratio, that a degree ranking supplies 80\% of the
above-chance signal, is a point estimate. And the whole analysis rests on one endpoint.

**Contents**

1. Environment and shared definitions
2. Data and evaluation set
3. Twenty GAT and twenty GCN models
4. The association with degree, by architecture
5. Agreement against performance, pooled
6. Intervals for the structural rankings
7. Intervals for the degree baseline and the ratio
8. A second dataset
9. Summary

*Approximate runtime: three to four hours on a T4. Each stage checkpoints.*

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part12"
for d in (R, EXPL, MODELS, CKPT): os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT)

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/ames_pipeline/ckpt_part12


The shared definitions are those used throughout, written out here so the notebook runs on a
fresh session.

In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


## 2. Data and evaluation set

In [4]:
import sys, json, time, glob
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from sklearn.metrics import roc_auc_score
from scipy import stats

SEED, PRIMARY_SPLIT_SEED = 42, 101
N_MODELS = 20
K_FRAC, K_GRID, N_IG_STEPS, N_TIEBREAK = 0.30, [0.10,0.20,0.30,0.40,0.50], 32, 200
N_BOOT = 10000
HP_GNN = json.load(open(f"{R}/hpo_gnn.json"))

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])
y_train = df_train["label"].values; y_test = df_test["label"].values
train_set, valid_set, test_set = (build_dataset(df_train), build_dataset(df_valid),
                                  build_dataset(df_test))
test_loader = DataLoader(test_set, batch_size=256)
POS_W = torch.tensor([(y_train==0).sum()/max((y_train==1).sum(),1)],
                     dtype=torch.float, device=DEVICE)
print(f"train {len(train_set)}, test {len(test_set)}; device {DEVICE}")

train 5804, test 726; device cuda


In [5]:
TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
assert all(p is not None for p in PAT.values())

REGIMES = {"full": list(TOX),
           "no_polycyclic": [k for k in TOX if k != "polycyclic_aromatic"],
           "polycyclic_only": ["polycyclic_aromatic"]}

def build_eval(frame, alerts):
    pats = {k: PAT[k] for k in alerts}
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

EVAL, CHANCE = build_eval(df_test, REGIMES["full"])
GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
N_EVAL = len(EVAL)

def overlap(ev, scores, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed); out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def lift(ov, ch):
    m = ~np.isnan(ov)
    return float(ov[m].mean()/ch[m].mean()) if m.sum() else np.nan

def scored(ev, ch, scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0):
    return np.array([lift(overlap(ev, scores, k_frac, base+j), ch) for j in range(n)])

def degree_scores(ev):
    return [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in ev]

DEG = degree_scores(EVAL)
DEG_LIFT = scored(EVAL, CHANCE, DEG, base=4000).mean()
ORACLE = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in EVAL]
CEILING = lift(overlap(EVAL, ORACLE, K_FRAC, 0), CHANCE)
print(f"evaluation set {N_EVAL}; degree {DEG_LIFT:.4f}; ceiling {CEILING:.4f}")

evaluation set 237; degree 1.2136; ceiling 1.9144


## 3. Twenty GAT and twenty GCN models

The same protocol as the GIN models already fitted. Weights and raw attribution scores are
retained, so the per-molecule analyses of Section 4 need no refitting.

In [6]:
def grad_scores(model, graphs, method="ig", n_steps=N_IG_STEPS, batch_size=64):
    """Gradient state and training mode are saved and restored."""
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
                xa = (x0*al).detach().requires_grad_(True)
                lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(lg.sum(), xa)[0]
            attr = x0*(total/n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            g = torch.autograd.grad(lg.sum(), xa)[0]
            attr = g if method == "saliency" else x0*g
        sc = attr.abs().sum(dim=1).detach().cpu().numpy(); b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(sc[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

STATE = f"{CKPT}/arch_models.csv"
meta = pd.read_csv(STATE).to_dict("records") if os.path.exists(STATE) else []
done = {(r["arch"], int(r["seed"])) for r in meta}
t0 = time.time()
for arch, base in [("gat", 30000), ("gcn", 40000)]:
    hp = HP_GNN[arch]
    for i in range(N_MODELS):
        sd = base + i
        if (arch, sd) in done: continue
        m, va = train_gnn(arch, hp, sd, train_set, valid_set, POS_W)
        torch.save({"state_dict": m.state_dict(), "valid_auc": float(va)},
                   f"{MODELS}/{arch}_attr_seed{sd}.pt")
        y, p = evaluate(m, test_loader)
        rec = {"arch": arch, "seed": sd, "valid_auc": float(va),
               "test_roc_auc": roc_auc_score(y, p)}
        sc = grad_scores(m, GRAPHS, "ig")
        np.savez_compressed(f"{EXPL}/scores_ig_{arch}_seed{sd}.npz",
                            **{f"m{j}": s for j, s in enumerate(sc)})
        for k in K_GRID:
            rec[f"lift_ig_k{int(k*100)}"] = scored(EVAL, CHANCE, sc, k, n=50, base=sd).mean()
        # per-molecule quantities for the mechanism analysis
        ov = np.nanmean([overlap(EVAL, sc, K_FRAC, sd*7+j) for j in range(50)], axis=0)
        np.save(f"{EXPL}/overlaps_{arch}_seed{sd}.npy", ov)
        rec["spearman_ig_degree"] = float(np.nanmean(
            [stats.spearmanr(s, d).correlation if len(set(d)) > 1 else np.nan
             for s, d in zip(sc, DEG)]))
        meta.append(rec); pd.DataFrame(meta).to_csv(STATE, index=False)
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  {arch} {sd}  test {rec['test_roc_auc']:.4f}  lift {rec['lift_ig_k30']:.4f}"
              f"  rho {rec['spearman_ig_degree']:+.3f}  ({(time.time()-t0)/60:.0f} min)")

meta = pd.DataFrame(meta)
print()
print(meta.groupby("arch")[["test_roc_auc","lift_ig_k30","spearman_ig_degree"]]
          .agg(["mean","std","count"]).round(4).to_string())

  gat 30000  test 0.7872  lift 1.2202  rho +0.697  (2 min)
  gat 30001  test 0.7886  lift 1.2052  rho +0.543  (4 min)
  gat 30002  test 0.7728  lift 1.2681  rho +0.340  (6 min)
  gat 30003  test 0.7701  lift 1.1901  rho +0.735  (8 min)
  gat 30004  test 0.7951  lift 1.1362  rho +0.740  (12 min)
  gat 30005  test 0.7801  lift 1.2305  rho +0.778  (14 min)
  gat 30006  test 0.7665  lift 1.1544  rho +0.391  (16 min)
  gat 30007  test 0.7685  lift 1.1348  rho +0.704  (18 min)
  gat 30008  test 0.7838  lift 1.1863  rho +0.719  (22 min)
  gat 30009  test 0.7771  lift 1.1346  rho +0.292  (24 min)
  gat 30010  test 0.7586  lift 1.1273  rho +0.089  (25 min)
  gat 30011  test 0.7812  lift 1.2736  rho +0.432  (28 min)
  gat 30012  test 0.7726  lift 1.2264  rho +0.746  (31 min)
  gat 30013  test 0.8031  lift 1.2094  rho +0.519  (34 min)
  gat 30014  test 0.7954  lift 1.2579  rho +0.618  (37 min)
  gat 30015  test 0.7858  lift 1.1722  rho +0.735  (40 min)
  gat 30016  test 0.7764  lift 1.3112  rho +

## 4. The association with degree, by architecture

For each architecture: the mean agreement, a test against the degree baseline, the
within-molecule association between attribution and degree, and the per-molecule correlation
with the degree ranking. The GIN column is recomputed from the stored scores of Part 9 so
that all three are treated identically.

In [7]:
def gin_records():
    g = pd.read_csv(f"{R}/table42_model_level_attribution.csv")
    rows = []
    for _, r in g.iterrows():
        p = f"{EXPL}/scores_ig_seed{int(r.seed)}.npz"
        rho = np.nan
        if os.path.exists(p):
            z = np.load(p); sc = [z[f"m{i}"] for i in range(N_EVAL)]
            rho = float(np.nanmean([stats.spearmanr(s, d).correlation if len(set(d)) > 1 else np.nan
                                    for s, d in zip(sc, DEG)]))
        rows.append({"arch": "gin", "seed": int(r.seed), "test_roc_auc": r.test_roc_auc,
                     "lift_ig_k30": r.lift_ig_k30, "spearman_ig_degree": rho})
    return pd.DataFrame(rows)

ALL = pd.concat([gin_records(), meta], ignore_index=True)
ALL.to_csv(f"{R}/table59_architecture_models.csv", index=False)

deg_ov = np.load(f"{EXPL}/degree_overlaps_primary.npy")
un_ov  = np.nanmean(np.load(f"{EXPL}/untrained_overlaps_gin.npy"), axis=0)
rows = []
for arch in ["gin", "gat", "gcn"]:
    v = ALL[ALL.arch == arch].lift_ig_k30.values
    t, p = stats.ttest_1samp(v, DEG_LIFT)
    lo, hi = stats.t.interval(0.95, len(v)-1, loc=v.mean(), scale=stats.sem(v))
    ovs = [np.load(f"{EXPL}/overlaps_{arch}_seed{int(s)}.npy")
           for s in ALL[ALL.arch == arch].seed
           if os.path.exists(f"{EXPL}/overlaps_{arch}_seed{int(s)}.npy")]
    r_deg = np.nan
    if ovs:
        mo = np.nanmean(ovs, axis=0); msk = ~(np.isnan(mo) | np.isnan(deg_ov))
        r_deg = float(stats.pearsonr(mo[msk], deg_ov[msk])[0])
    rows.append({"architecture": arch.upper(), "n_models": len(v),
                 "lift_mean": v.mean(), "lift_sd": v.std(ddof=1),
                 "ci_low": lo, "ci_high": hi,
                 "t_vs_degree": t, "p_vs_degree": p,
                 "pct_above_degree": 100*(v > DEG_LIFT).mean(),
                 "spearman_ig_degree": ALL[ALL.arch == arch].spearman_ig_degree.mean(),
                 "r_with_degree_overlap": r_deg,
                 "degree_share_pct": 100*(DEG_LIFT-1)/(v.mean()-1)})
arch_tab = pd.DataFrame(rows)
arch_tab.to_csv(f"{R}/table60_architecture_summary.csv", index=False)
print(arch_tab.round(4).to_string(index=False))
print(f"\n  degree baseline {DEG_LIFT:.4f}; untrained GIN "
      f"{pd.read_csv(f'{R}/table12_untrained_null.csv').set_index('arch').loc['gin','mean']:.4f}")

architecture  n_models  lift_mean  lift_sd  ci_low  ci_high  t_vs_degree  p_vs_degree  pct_above_degree  spearman_ig_degree  r_with_degree_overlap  degree_share_pct
         GIN        20     1.2655   0.0916  1.2226   1.3084       2.5358       0.0202              75.0              0.4292                    NaN           80.4383
         GAT        20     1.2026   0.0529  1.1778   1.2273      -0.9288       0.3647              40.0              0.5685                 0.9550          105.4261
         GCN        20     1.1531   0.0766  1.1173   1.1890      -3.5265       0.0023              25.0              0.4963                 0.8888          139.4623

  degree baseline 1.2136; untrained GIN 1.1156


## 5. Agreement against performance, pooled

Twenty models were too few to resolve whether agreement tracks predictive performance. Sixty
models across three architectures narrow the interval, and the question can also be asked
within each architecture.

In [8]:
def fisher(r, n):
    z = np.arctanh(r); se = 1/np.sqrt(n-3)
    return np.tanh(z-1.96*se), np.tanh(z+1.96*se)

rows = []
for arch in ["gin", "gat", "gcn", "all"]:
    s = ALL if arch == "all" else ALL[ALL.arch == arch]
    s = s.dropna(subset=["test_roc_auc", "lift_ig_k30"])
    r, p = stats.pearsonr(s.test_roc_auc, s.lift_ig_k30)
    lo, hi = fisher(r, len(s))
    rows.append({"scope": arch, "n": len(s), "pearson_r": r, "ci_low": lo, "ci_high": hi,
                 "p": p, "roc_min": s.test_roc_auc.min(), "roc_max": s.test_roc_auc.max()})
perf = pd.DataFrame(rows)
perf.to_csv(f"{R}/table61_lift_vs_performance_pooled.csv", index=False)
print(perf.round(4).to_string(index=False))
print("\n  pooling across architectures mixes two sources of variation and is reported")
print("  alongside the within-architecture correlations rather than in place of them.")

scope  n  pearson_r  ci_low  ci_high      p  roc_min  roc_max
  gin 20     0.2557 -0.2107   0.6272 0.2766   0.7584   0.8013
  gat 20     0.2725 -0.1934   0.6381 0.2451   0.7586   0.8031
  gcn 20     0.2926 -0.1723   0.6508 0.2107   0.7508   0.7928
  all 60     0.3939  0.1555   0.5889 0.0018   0.7508   0.8031

  pooling across architectures mixes two sources of variation and is reported
  alongside the within-architecture correlations rather than in place of them.


## 6. Intervals for the structural rankings

The panel of rankings has been reported as point estimates under three regimes, and a claim
about which ranking stays above chance was read from them. Each cell is resampled over
molecules here, so the claim can be stated in terms of intervals or withdrawn.

In [9]:
HETERO = {"N","O","S","P","F","Cl","Br","I","Se","B","Si"}
def descriptor(ev, name):
    out = []
    for e in ev:
        ats = list(e["mol"].GetAtoms())
        if   name == "degree":            v = [a.GetDegree() for a in ats]
        elif name == "heteroatom":        v = [1.0 if a.GetSymbol() in HETERO else 0.0 for a in ats]
        elif name == "hetero_neighbours": v = [sum(1 for nb in a.GetNeighbors()
                                                   if nb.GetSymbol() in HETERO) for a in ats]
        elif name == "aromatic":          v = [1.0 if a.GetIsAromatic() else 0.0 for a in ats]
        elif name == "in_ring":           v = [1.0 if a.IsInRing() else 0.0 for a in ats]
        elif name == "abs_formal_charge": v = [abs(a.GetFormalCharge()) for a in ats]
        elif name == "hydrogens":         v = [a.GetTotalNumHs() for a in ats]
        elif name == "hetero_plus_degree":
            v = [(2.0 if a.GetSymbol() in HETERO else 0.0)+0.25*a.GetDegree() for a in ats]
        elif name == "hetero_or_neighbour":
            v = [1.0 if (a.GetSymbol() in HETERO or
                 any(nb.GetSymbol() in HETERO for nb in a.GetNeighbors())) else 0.0 for a in ats]
        elif name == "rarity":
            key = [(a.GetSymbol(), a.GetDegree(), a.GetIsAromatic()) for a in ats]
            cnt = {k: key.count(k) for k in set(key)}
            v = [1.0/cnt[k] for k in key]
        elif name == "degree_deviation":
            d = np.array([a.GetDegree() for a in ats], dtype=float); v = list(np.abs(d-d.mean()))
        else: raise ValueError(name)
        out.append(np.asarray(v, dtype=float))
    return out

RANKINGS = ["degree","hetero_neighbours","degree_deviation","hetero_plus_degree",
            "abs_formal_charge","heteroatom","hetero_or_neighbour","rarity",
            "aromatic","in_ring","hydrogens"]

def boot_lift(ov, ch, n_boot=N_BOOT, seed=SEED):
    """Resample molecules; ov is the tie-break mean overlap per molecule."""
    rng = np.random.default_rng(seed); n = len(ov); out = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.integers(0, n, n)
        o, c = ov[i], ch[i]; m = ~np.isnan(o)
        out[b] = o[m].mean()/c[m].mean() if m.sum() else np.nan
    return out

RB_PATH = f"{R}/table62_regime_intervals.csv"
if os.path.exists(RB_PATH):
    reg_ci = pd.read_csv(RB_PATH); print("loaded from checkpoint")
else:
    rows = []
    for reg, alerts in REGIMES.items():
        ev, ch = build_eval(df_test, alerts)
        orc = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in ev]
        ceil = lift(overlap(ev, orc, K_FRAC, 0), ch)
        for name in RANKINGS + ["random"]:
            if name == "random":
                sc = [np.random.default_rng(7000+i).random(e["n_atoms"]) for i, e in enumerate(ev)]
            else:
                sc = descriptor(ev, name)
            ov = np.nanmean([overlap(ev, sc, K_FRAC, 1000+j) for j in range(N_TIEBREAK)], axis=0)
            v = boot_lift(ov, ch)
            rows.append({"regime": reg, "ranking": name, "n_eval": len(ev),
                         "chance": ch.mean(), "ceiling": ceil,
                         "lift": lift(ov, ch), "ci_low": np.nanpercentile(v, 2.5),
                         "ci_high": np.nanpercentile(v, 97.5),
                         "excludes_chance": bool(np.nanpercentile(v, 2.5) > 1.0)})
        print(f"  {reg}: done")
    reg_ci = pd.DataFrame(rows); reg_ci.to_csv(RB_PATH, index=False)

piv = reg_ci.pivot_table(index="ranking", columns="regime", values="lift")
exc = reg_ci.pivot_table(index="ranking", columns="regime", values="excludes_chance")
print()
for rk in RANKINGS:
    cells = " | ".join(
        f"{reg_ci[(reg_ci.ranking==rk)&(reg_ci.regime==g)].lift.iloc[0]:.3f}"
        f" [{reg_ci[(reg_ci.ranking==rk)&(reg_ci.regime==g)].ci_low.iloc[0]:.3f},"
        f"{reg_ci[(reg_ci.ranking==rk)&(reg_ci.regime==g)].ci_high.iloc[0]:.3f}]"
        for g in ["full","no_polycyclic","polycyclic_only"])
    n_exc = int(exc.loc[rk].sum())
    print(f"  {rk:20s} {cells}   above chance in {n_exc}/3")
survivors = [rk for rk in RANKINGS if int(exc.loc[rk].sum()) == 3]
print(f"\n  rankings whose interval excludes chance under all three regimes: {survivors}")

  full: done
  no_polycyclic: done
  polycyclic_only: done

  degree               1.214 [1.184,1.245] | 1.278 [1.223,1.334] | 1.162 [1.127,1.198]   above chance in 3/3
  hetero_neighbours    1.163 [1.112,1.222] | 1.679 [1.585,1.778] | 0.812 [0.777,0.845]   above chance in 2/3
  degree_deviation     1.130 [1.070,1.194] | 1.454 [1.348,1.561] | 0.899 [0.843,0.954]   above chance in 2/3
  hetero_plus_degree   1.092 [1.039,1.149] | 1.534 [1.441,1.635] | 0.786 [0.727,0.840]   above chance in 2/3
  abs_formal_charge    1.067 [1.043,1.093] | 1.170 [1.115,1.228] | 0.992 [0.983,0.999]   above chance in 2/3
  heteroatom           1.059 [1.010,1.114] | 1.513 [1.438,1.592] | 0.744 [0.697,0.784]   above chance in 2/3
  hetero_or_neighbour  1.055 [0.993,1.124] | 1.735 [1.630,1.848] | 0.603 [0.549,0.656]   above chance in 1/3
  rarity               0.926 [0.853,1.004] | 1.599 [1.458,1.742] | 0.481 [0.408,0.549]   above chance in 1/3
  aromatic             0.982 [0.929,1.033] | 0.519 [0.445,0.588] | 1

## 7. Intervals for the degree baseline and the ratio

The agreement of the degree ranking has so far carried only the spread over tie-breaks, which
omits sampling of molecules. The ratio of above-chance signal, quoted as a headline, has
carried no interval at all. Both are resampled here over molecules, the trained models being
resampled with them.

In [10]:
deg_ov_tb = np.nanmean([overlap(EVAL, DEG, K_FRAC, 5000+j) for j in range(N_TIEBREAK)], axis=0)
v_deg = boot_lift(deg_ov_tb, CHANCE)
gin_lifts = ALL[ALL.arch == "gin"].lift_ig_k30.values

rng = np.random.default_rng(SEED)
ratio = np.empty(N_BOOT); n = len(EVAL)
gin_ovs = [np.load(f"{EXPL}/trained_overlaps.npz")["overlaps"]]
TR = gin_ovs[0]
for b in range(N_BOOT):
    i = rng.integers(0, n, n)
    o, c = deg_ov_tb[i], CHANCE[i]; m = ~np.isnan(o)
    d = o[m].mean()/c[m].mean()
    ti = rng.integers(0, TR.shape[0], TR.shape[0])
    to = np.nanmean(TR[np.ix_(ti, i)], axis=0); mt = ~np.isnan(to)
    t = to[mt].mean()/c[mt].mean()
    ratio[b] = (d-1)/(t-1) if t > 1 else np.nan

rows = [{"quantity": "degree lift", "estimate": lift(deg_ov_tb, CHANCE),
         "ci_low": np.nanpercentile(v_deg, 2.5), "ci_high": np.nanpercentile(v_deg, 97.5)},
        {"quantity": "degree share of trained above-chance signal (%)",
         "estimate": 100*np.nanmedian(ratio),
         "ci_low": 100*np.nanpercentile(ratio, 2.5),
         "ci_high": 100*np.nanpercentile(ratio, 97.5)}]
t, p = stats.ttest_1samp(gin_lifts, lift(deg_ov_tb, CHANCE))
rows.append({"quantity": "trained GIN mean vs degree, t(19)", "estimate": t,
             "ci_low": np.nan, "ci_high": p})
ratio_tab = pd.DataFrame(rows)
ratio_tab.to_csv(f"{R}/table63_degree_intervals.csv", index=False)
print(ratio_tab.round(4).to_string(index=False))
print(f"\n  point ratio from the means: "
      f"{100*(lift(deg_ov_tb,CHANCE)-1)/(gin_lifts.mean()-1):.1f}%")

                                       quantity  estimate  ci_low  ci_high
                                    degree lift    1.2117  1.1826   1.2432
degree share of trained above-chance signal (%)   87.2546 69.3547 114.3720
              trained GIN mean vs degree, t(19)    2.6255     NaN   0.0167

  point ratio from the means: 79.7%


## 8. A second dataset

The mechanism claim, that an untrained network's attributions reproduce a degree ranking,
needs no labels and no alert library: it is a statement about what message passing does to
gradients on molecular graphs. It can therefore be tested on any set of molecules. If the
correlation holds elsewhere, the claim is not specific to this endpoint.

The loader tries several sources in turn and falls back to the training molecules of this
dataset, which are disjoint from the evaluation set used everywhere else.

In [11]:
SECOND = None
try:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyTDC"],
                   check=False, capture_output=True, timeout=600)
    from tdc.single_pred import Tox
    d2 = Tox(name="hERG").get_data()
    SECOND = ("hERG (TDC)", d2["Drug"].tolist())
except Exception as e1:
    print("TDC unavailable:", type(e1).__name__)

if SECOND is None:
    for url, name in [
        ("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv", "BBBP"),
        ("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv", "ClinTox")]:
        try:
            t = pd.read_csv(url)
            col = "smiles" if "smiles" in t.columns else t.columns[-1]
            SECOND = (name, t[col].dropna().astype(str).tolist())
            break
        except Exception as e2:
            print(f"{name} unavailable:", type(e2).__name__)

if SECOND is None:
    SECOND = ("Ames training molecules", df_train["canonical"].tolist())
print("second set:", SECOND[0], "|", len(SECOND[1]), "molecules")

TDC unavailable: ModuleNotFoundError
second set: BBBP | 2050 molecules


In [12]:
name2, smiles2 = SECOND
rng = np.random.default_rng(SEED)
pool = [s for s in smiles2 if Chem.MolFromSmiles(s) is not None]
pool = list(dict.fromkeys(pool))
sel = [pool[i] for i in rng.permutation(len(pool))[:min(300, len(pool))]]
EV2 = []
for s in sel:
    mol = Chem.MolFromSmiles(s)
    if mol is None or mol.GetNumAtoms() < 4: continue
    EV2.append({"smiles": s, "mol": mol, "tox": set(), "n_atoms": mol.GetNumAtoms()})
GR2 = [smiles_to_data(e["smiles"], 1.0) for e in EV2]
DEG2 = degree_scores(EV2)
print(f"{len(EV2)} molecules, mean {np.mean([e['n_atoms'] for e in EV2]):.1f} atoms")

def rank_corr(scores, ref):
    """Mean within-molecule Spearman between two atom rankings."""
    v = [stats.spearmanr(s, d).correlation if len(set(d)) > 1 and len(set(np.round(s, 12))) > 1
         else np.nan for s, d in zip(scores, ref)]
    return float(np.nanmean(v)), float(np.nanmean(np.array(v) > 0))

SEC_PATH = f"{R}/table64_second_dataset.csv"
if os.path.exists(SEC_PATH):
    sec = pd.read_csv(SEC_PATH); print("loaded from checkpoint")
else:
    N_DRAWS = 200; hp = HP_GNN["gin"]
    rows = []
    for label, ev, gr, dg in [("Ames test (evaluation set)", EVAL, GRAPHS, DEG),
                              (name2, EV2, GR2, DEG2)]:
        rs, ps = [], []
        t0 = time.time()
        for j in range(N_DRAWS):
            set_seed(96000+j)
            m = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"],
                          hp.get("residual", True))
            r, pos = rank_corr(grad_scores(m, gr, "ig"), dg)
            rs.append(r); ps.append(pos)
            del m
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        rows.append({"dataset": label, "n_molecules": len(ev), "n_draws": N_DRAWS,
                     "spearman_mean": float(np.mean(rs)), "spearman_sd": float(np.std(rs, ddof=1)),
                     "pct_molecules_positive": 100*float(np.mean(ps)),
                     "minutes": (time.time()-t0)/60})
        print(f"  {label}: rho {rows[-1]['spearman_mean']:+.3f}, "
              f"{rows[-1]['pct_molecules_positive']:.0f}% positive "
              f"({rows[-1]['minutes']:.0f} min)")
    sec = pd.DataFrame(rows); sec.to_csv(SEC_PATH, index=False)
print()
print(sec.round(4).to_string(index=False))

299 molecules, mean 24.6 atoms
  Ames test (evaluation set): rho +0.685, 98% positive (3 min)
  BBBP: rho +0.646, 99% positive (4 min)

                   dataset  n_molecules  n_draws  spearman_mean  spearman_sd  pct_molecules_positive  minutes
Ames test (evaluation set)          237      200         0.6852       0.1045                 98.4789   3.1710
                      BBBP          299      200         0.6459       0.1019                 98.9649   3.5821


## 9. Summary

In [13]:
print("architectures")
print(arch_tab[["architecture","n_models","lift_mean","p_vs_degree","spearman_ig_degree",
                "degree_share_pct"]].round(4).to_string(index=False))
print("\nagreement against performance")
print(perf[["scope","n","pearson_r","ci_low","ci_high"]].round(3).to_string(index=False))
print("\nrankings above chance in all three regimes:", survivors)
print("\ndegree baseline and ratio")
print(ratio_tab.round(4).to_string(index=False))
print("\nsecond dataset")
print(sec[["dataset","n_molecules","spearman_mean","pct_molecules_positive"]].round(3).to_string(index=False))
print("\nWritten to", R)
for f in ["table59_architecture_models.csv","table60_architecture_summary.csv",
          "table61_lift_vs_performance_pooled.csv","table62_regime_intervals.csv",
          "table63_degree_intervals.csv","table64_second_dataset.csv"]:
    print("  -", f)
import shutil
from google.colab import files
shutil.make_archive("/content/part12_results", "zip", R)
files.download("/content/part12_results.zip")

architectures
architecture  n_models  lift_mean  p_vs_degree  spearman_ig_degree  degree_share_pct
         GIN        20     1.2655       0.0202              0.4292           80.4383
         GAT        20     1.2026       0.3647              0.5685          105.4261
         GCN        20     1.1531       0.0023              0.4963          139.4623

agreement against performance
scope  n  pearson_r  ci_low  ci_high
  gin 20      0.256  -0.211    0.627
  gat 20      0.272  -0.193    0.638
  gcn 20      0.293  -0.172    0.651
  all 60      0.394   0.155    0.589

rankings above chance in all three regimes: ['degree']

degree baseline and ratio
                                       quantity  estimate  ci_low  ci_high
                                    degree lift    1.2117  1.1826   1.2432
degree share of trained above-chance signal (%)   87.2546 69.3547 114.3720
              trained GIN mean vs degree, t(19)    2.6255     NaN   0.0167

second dataset
                   dataset  n_m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>